# HW1 Tutorial: Building a Gymnasium Environment for Underwater Caldera Exploration

In this notebook, we will turn a science mission into a simualted environment.

Our setting is an **underwater caldera exploration mission**. An autonomous vehicle moves through a volcanic basin, gathering measurementsusing its limited fuel.
It t

and tries to spend its limited budget on the most informative samples. It's objective is to try and locate the deepest point in the caldera.




<img src="images/Kolumbo-submarine-volcano-and-caldera-Alexandri-et-al.jpg" alt="Kolumbo submarine caldera" width="500" />


The core learning idea is **adaptive sampling**:

- The agent should not sample everywhere equally.
- It should focus on locations where uncertainty is high or where interesting chemistry is likely.
- Each move and each sample consumes time or energy.

By the end of the notebook, you should understand how to design:

- the observation space
- the action space
- the `reset()` method
- the `step()` method
- a reward that encourages useful exploration


## What is Gymnasium and why are we using it?



Gymnasium is a standard library for building reinforcement learning environments. It gives us a clear structure for describing how an agent interacts with a world over time.

A Gymnasium environment usually defines:

- an **observation space** describing what the agent can see
- an **action space** describing what the agent can do
- a `reset()` method for starting a new episode
- a `step(action)` method for applying one action and returning the result

Each call to `step()` returns:

```python
observation, reward, terminated, truncated, info
```

We are using Gymnasium here because it gives our underwater exploration problem a clean, reusable RL interface that will work well for simulation, debugging, and later training experiments.


## What are we building?

In this assignment, we are building a custom **Gymnasium environment** for an underwater caldera exploration mission.

That means we need to define:

- what the agent observes about the mission state
- what actions it can take while exploring
- what reward it receives for useful behavior
- when an episode should end

By representing the mission as a Gymnasium environment, we can turn the exploration problem into a clean RL task that is easy to simulate, test, and train on.

## Mission story

Imagine a robotic submersible exploring a 2D grid laid over an underwater caldera.

At each time step it can:

- move north
- move south
- move east
- move west
- take a sample at its current location

Some grid cells are more scientifically valuable than others. For example, hydrothermal vents or chemically unusual regions may produce a larger reward when sampled.

The agent should balance two competing goals:

- cover the map efficiently
- spend sampling effort where it matters most


## Designing the observation space

A valuable observation should contain the information the agent needs to make a decision.

For this mission, a simple observation might include:

- current row position
- current column position
- remaining energy budget
- whether the current cell has already been sampled

That is enough for a small tutorial environment.

A more advanced environment could also include:

- an uncertainty estimate
- local sensor readings
- a map of visited cells
- a compressed belief state over the whole caldera

For now, we will keep the observation compact and easy to implement.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

example_observation_space = spaces.Dict(
    {
        "position": spaces.Box(low=0, high=4, shape=(2,), dtype=int),
        "energy": spaces.Discrete(21),
        "sampled_here": spaces.Discrete(2),
    }
)

example_action_space = spaces.Discrete(5)

example_observation_space, example_action_space


## Designing the action space

Our action space is naturally discrete.

We can encode actions as:

- `0`: north
- `1`: south
- `2`: west
- `3`: east
- `4`: sample

This is a good fit for `spaces.Discrete(5)`.

## Reward design for adaptive sampling

Reward design is where the mission objective becomes explicit.

A simple reward function could be:

- small penalty for movement, to represent energy cost
- larger positive reward for sampling a valuable unsampled cell
- small penalty for sampling a location twice
- episode ends when the energy budget runs out

For adaptive sampling, we want reward to prefer *informative* samples instead of random ones.

One simple approximation is to store a hidden value map:

- ordinary cells give low reward
- scientifically interesting cells give high reward

Then the agent learns that the best policy is not just to move, but to move toward useful places and sample selectively.

In [ ]:
import numpy as np

value_map = np.array(
    [
        [1, 1, 2, 1, 1],
        [1, 2, 4, 2, 1],
        [1, 3, 8, 3, 1],
        [1, 2, 4, 2, 1],
        [1, 1, 2, 1, 1],
    ],
    dtype=float,
)

value_map


## Implementation plan for `CalderaEnv`

Open [to_implement.py](C:/Users/sarahk/Documents/PYCHARM-projects/PARLAI/hw/hw1/to_implement.py). The class already has the right method names. We just need to define the environment details.

A reasonable implementation plan is:

1. Store mission parameters such as grid size and max energy.
2. Define `observation_space` and `action_space`.
3. In `reset()`, place the robot at a start cell and clear sampled locations.
4. In `step()`, update the state based on the chosen action.
5. Compute a reward and decide whether the episode is over.


In [ ]:
from hw.hw1.to_implement import CalderaEnv

print(CalderaEnv)


## What should `reset()` do?

At the start of each episode, we want a fresh mission.

`reset()` should usually:

- restore the robot to a starting location
- reset the remaining energy
- clear the sampled-cell memory
- return the first observation and an `info` dictionary

A typical observation might look like:

```python
{
    "position": np.array([0, 0]),
    "energy": 20,
    "sampled_here": 0,
}
```

## What should `step()` do?

Each step should:

- read the chosen action
- move or sample
- reduce energy
- compute reward
- decide whether the episode is terminated or truncated
- return the next observation and extra info

A clean mental model is:

```python
take action -> update state -> compute reward -> package outputs
```

## Pseudocode

Here is a simple version of the environment logic:

```python
if action is movement:
    move within grid bounds
    reward = -0.1

elif action is sample:
    if current cell was not sampled before:
        reward = scientific_value_of_cell
        mark sampled
    else:
        reward = -0.5

energy -= 1
done = energy == 0
```

## Suggested implementation sketch

The sketch below is not meant to replace your file directly. It is here to show the structure you are aiming for.

In [ ]:
tutorial_code = '''
from typing import Any

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class AGymEnv(gym.Env):
    def __init__(self, a_string: str):
        self.a_string = a_string
        self.grid_size = 5
        self.max_energy = 20
        self.value_map = np.array([
            [1, 1, 2, 1, 1],
            [1, 2, 4, 2, 1],
            [1, 3, 8, 3, 1],
            [1, 2, 4, 2, 1],
            [1, 1, 2, 1, 1],
        ], dtype=float)

        self.observation_space = spaces.Dict({
            "position": spaces.Box(low=0, high=self.grid_size - 1, shape=(2,), dtype=int),
            "energy": spaces.Discrete(self.max_energy + 1),
            "sampled_here": spaces.Discrete(2),
        })

        self.action_space = spaces.Discrete(5)

    def _get_obs(self):
        sampled_here = int(tuple(self.position) in self.sampled_cells)
        return {
            "position": self.position.copy(),
            "energy": self.energy,
            "sampled_here": sampled_here,
        }

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.position = np.array([0, 0], dtype=int)
        self.energy = self.max_energy
        self.sampled_cells = set()
        return self._get_obs(), {}

    def step(self, action: int):
        reward = -0.1
        row, col = self.position

        if action == 0:
            row = max(0, row - 1)
        elif action == 1:
            row = min(self.grid_size - 1, row + 1)
        elif action == 2:
            col = max(0, col - 1)
        elif action == 3:
            col = min(self.grid_size - 1, col + 1)
        elif action == 4:
            cell = tuple(self.position)
            if cell not in self.sampled_cells:
                self.sampled_cells.add(cell)
                reward = float(self.value_map[cell])
            else:
                reward = -0.5

        self.position = np.array([row, col], dtype=int)
        self.energy -= 1

        terminated = self.energy <= 0
        truncated = False
        info = {"mission": self.a_string}

        return self._get_obs(), reward, terminated, truncated, info
'''

print(tutorial_code)


## Why this environment is adaptive

The environment becomes adaptive because the agent gets different value from different sample locations.

If the reward is larger near scientifically interesting parts of the caldera, then the optimal strategy is to:

- travel toward promising areas
- avoid wasting samples on repeated cells
- make good use of a limited energy budget

That is the main idea of adaptive sampling in a compact RL environment.

## Possible extensions

Once the basic version works, you can make the mission more realistic.

Possible extensions include:

- stochastic ocean currents that push the robot
- noisy sensor readings
- a hidden contamination plume
- partial observability
- bonus reward for discovering new high-value regions
- penalties for revisiting too often

These extensions make the environment richer without changing the basic Gymnasium interface.

## Your task

Use this notebook as a guide while implementing [to_implement.py](C:/Users/sarahk/Documents/PYCHARM-projects/PARLAI/hw/hw1/to_implement.py).

A good next step is to fill in:

- `self.observation_space`
- `self.action_space`
- `reset()`
- `step()`

If you want, we can do the next part together and I can also implement `AGymEnv` in the Python file for you.